In [ ]:
from selenium import webdriver
import pandas as pd
import time
import os
import numpy as np
pd.set_option('display.max_colwidth', None) # stop truncating long text
url = "https://www.goodreads.com/list/best_of_year/2025"

Data Collection / Mining

In [181]:
df = pd.DataFrame(columns=["book_title","publication_date","author","genres","ratings",
                           "num_pages","rank","language","curr_readers","want_read"])
# scrape page
driver = webdriver.Chrome()
for i in range(1,7):
    driver.get(f'https://www.goodreads.com/list/best_of_year/2025?id=222558.Best_Books_of_2025&page={i}')
    time.sleep(.5)

    # close popup on list page if it exists
    try:
        popup = driver.find_element('xpath','/html/body/div[3]/div/div/div[1]/button/img')
        popup.click()
        print("popup closed")
    except:
        print("no popup")
        pass

    books = driver.find_elements('xpath', "//tr[contains(@itemtype, 'schema.org/Book')]")

    # extract list-page info
    book_data = []
    for book in books:
        rank = book.find_element('xpath', ".//td[contains(@class, 'number')]").text
        book_title = book.find_element('xpath', ".//a[contains(@class, 'bookTitle')]").text
        ratings = book.find_element('xpath', ".//span[contains(@class, 'minirating')]").text
        author = book.find_element('xpath', ".//a[contains(@class, 'authorName')]").text
        book_link = book.find_element('xpath', ".//a[contains(@class, 'bookTitle')]").get_attribute("href")
        
        book_data.append({
            "rank": rank,
            "book_title": book_title,
            "ratings": ratings,
            "author": author,
            "book_link": book_link
        })

    # visit each book link
    for item in book_data:
        driver.get(item["book_link"])
        time.sleep(.5)

        # close popup on book page if it exists
        try:
            popup = driver.find_element('xpath','/html/body/div[3]/div/div[1]/div/div/button')
            popup.click()
            print("popup closed")
        except:
            print("no popup")
            pass
        try:
            publication_date = driver.find_element('xpath','//p[contains(@data-testid, "publicationInfo")]').text
        except:
            publication_date =""
        try:
            genres = driver.find_elements('xpath','//span[contains(@class, "BookPageMetadataSection__genreButton")]')
            genres = [genre.text for genre in genres]
            genres = ", ".join(genres)
        except:
            genres = ""
        try:
            num_pages = driver.find_element('xpath','//p[contains(@data-testid, "pagesFormat")]').text
        except:
            num_pages = ""
            
        try:
            more_button = driver.find_element('xpath','//button[contains(@aria-label, "Book details and editions")]')
            more_button.click()
            time.sleep(.5)
        except:
            pass
        try:
            language = driver.find_element('xpath',"//dt[normalize-space()='Language']/following-sibling::dd[1]//div[@data-testid='contentContainer']").text
            print("Language:", language)
        except:
            language =""
            print("Language not found.")
        try:
            num_reading = driver.find_element('xpath','//div[contains(@data-testid, "currentlyReadingSignal")]').text
        except:
            num_reading = ""
        try:
            num_want_read = driver.find_element('xpath','//div[contains(@data-testid, "toReadSignal")]').text
        except:
            num_want_read = ""
        print("finished scraping", item["book_title"])
        df.loc[len(df)] = [
            item["book_title"],
            publication_date,
            item["author"],
            genres,
            item["ratings"],
            num_pages,
            item["rank"],
            language,
            num_reading,
            num_want_read
        ]
driver.quit()

no popup
popup closed
Language not found.
finished scraping Sunrise on the Reaping (The Hunger Games, #0.5)
no popup
Language not found.
finished scraping Onyx Storm (The Empyrean, #3)
no popup
Language: English
finished scraping Atmosphere
no popup
Language: English
finished scraping My Friends
no popup
Language: English
finished scraping Wild Dark Shore
no popup
Language: English
finished scraping Deep End
no popup
Language: English
finished scraping Great Big Beautiful Life
no popup
Language not found.
finished scraping The Favorites
no popup
Language: English
finished scraping Rebel Witch (The Crimson Moth, #2)
no popup
Language not found.
finished scraping Everything Is Tuberculosis: The History and Persistence of Our Deadliest Infection
no popup
Language: English
finished scraping Broken Country
no popup
Language: English
finished scraping The Names
no popup
Language not found.
finished scraping Beg, Borrow, or Steal (When in Rome, #3)
no popup
Language not found.
finished scrapi

In [ ]:
# to avoid re-running the data mining code
df.to_csv("./data/raw_data.csv", index=False)

Data Wrangling / Cleaning

In [192]:
df = pd.read_csv("./data/raw_data.csv")
df["language"] = df["language"].replace(np.nan, "")

In [193]:
df["publication_date"] = df["publication_date"].astype(str).str[16:]
df["publication_date"] = pd.to_datetime(df["publication_date"], format="%B %d, %Y",errors="coerce")
df["avg_rating"] = df["ratings"].astype(str).str[:4]
df["avg_rating"] = pd.to_numeric(df["avg_rating"], errors="coerce")
df["num_ratings"]=df["ratings"].str.extract(r"([\d,]+)\s+ratings")
df["num_ratings"]=df["num_ratings"].str.replace(",","").astype(int)
df.drop("ratings", axis=1, inplace=True)
df["num_pages"] = df["num_pages"].str.extract(r"([\d,]+)\s+pages")[0].str.replace(",", "", regex=False).astype("Int64")
df["curr_readers"] = df["curr_readers"].str.extract(r"([\d]+)\s+people").astype("Int64")
df["want_read"] = df["want_read"].str.extract(r"([\d]+)\s+people").astype("Int64")

In [217]:
df.to_csv("data/data.csv", index=False)

EDA

In [216]:
df.head()

,book_title,publication_date,author,genres,num_pages,rank,language,curr_readers,want_read,avg_rating,num_ratings
0,"Sunrise on the Reaping (The Hunger Games, #0.5)",2025-03-18,Suzanne Collins,"Dystopia, Young Adult, Fantasy, Fiction, Audiobook, Science Fiction, Book Club",387,1,,71857,607177,4.50,1131555
1,"Onyx Storm (The Empyrean, #3)",2025-01-21,Rebecca Yarros,"Fantasy, Romantasy, Romance, Audiobook, Dragons, Fiction, Fantasy Romance",544,2,,211783,693630,4.21,1741280
2,Atmosphere,2025-06-03,Taylor Jenkins Reid,"Historical Fiction, Fiction, Romance, Audiobook, Book Club, LGBT, Historical",337,3,English,63050,816568,4.33,766849
3,My Friends,2025-05-06,Fredrik Backman,"Fiction, Book Club, Audiobook, Contemporary, Literary Fiction, Coming Of Age, Friendship",436,4,English,63867,711684,4.36,428065
4,Wild Dark Shore,2025-03-04,Charlotte McConaghy,"Fiction, Mystery, Book Club, Audiobook, Thriller, Mystery Thriller, Literary Fiction",298,5,English,44288,653895,4.10,404074


In [ ]:
df_genres = df[["book_title","genres", "avg_rating"]]
df_genres["genres"] = df_genres["genres"].str.split(", ")
df_genres = df_genres.explode("genres")

,book_title,genres,avg_rating
0,"Sunrise on the Reaping (The Hunger Games, #0.5)",Dystopia,4.50
0,"Sunrise on the Reaping (The Hunger Games, #0.5)",Young Adult,4.50
0,"Sunrise on the Reaping (The Hunger Games, #0.5)",Fantasy,4.50
0,"Sunrise on the Reaping (The Hunger Games, #0.5)",Fiction,4.50
0,"Sunrise on the Reaping (The Hunger Games, #0.5)",Audiobook,4.50
...,...,...,...
550,Retreat,Mystery Thriller,3.69
550,Retreat,Audiobook,3.69
550,Retreat,Fiction,3.69
550,Retreat,Suspense,3.69


In [ ]:
# Most highly-rated genres
genre_rating = df_genres.dropna(subset=["avg_rating"])
genre_rating = genre_rating.groupby(["genres"])["avg_rating"].mean().round(2).reset_index().sort_values(by=["avg_rating"], ascending=False)
genre_rating

,genres,avg_rating
198,Social Justice,4.59
64,Disability,4.58
91,Futuristic,4.53
225,Werewolves,4.52
24,Autobiography,4.46
...,...,...
132,Literature,3.56
179,Road Trip,3.56
41,College,3.48
100,Graphic Novels Manga,3.34


In [ ]:
# Most popular genres
df_genres["genres"].value_counts()

genres
Fiction                     396
Audiobook                   390
Romance                     195
Fantasy                     186
Mystery                     173
                           ... 
Young Adult Romance           1
African American Romance      1
Womens                        1
Monsters                      1
College                       1
Name: count, Length: 237, dtype: int64